In [2]:
"""
Normalizacion radiometrica para mosaico SAR (adaptacion practica del metodo RPRN,
Liu et al. 2022, IEEE GRSL, "Radiometric Principle-Based Radiometric Normalization
Method for SAR Images Mosaic").

Esta version es fiel al alcance del paper: SOLO corrige diferencias de MEDIA
(brillo) entre imagenes mediante un factor multiplicativo h_i por imagen. No se
intenta igualar la desviacion estandar/textura -- esa diferencia entre subswaths
adyacentes (IW1/IW2/IW3) responde en parte al angulo de incidencia distinto de
cada uno, que es fisicamente esperado y no debe eliminarse por la fuerza.

Adaptacion respecto al paper original:
- El paper trabaja con SLC sin geocodificar y usa RPCs + tie points para encontrar
  areas de cobertura consistente en espacio-objeto entre imagenes.
- Aqui las imagenes YA vienen con Terrain Correction (georreferenciadas), por lo que
  el area de solape se obtiene directamente de la interseccion geografica de cada
  GeoTIFF -- no hace falta paso de registro ni RPCs.
- El modelo de ajuste h_i * sigma_i = h_j * sigma_j se resuelve aqui en forma
  log-lineal en un solo paso de minimos cuadrados con regularizacion ridge
  (equivalente en el optimo al esquema iterativo del paper, pero sin iterar).

Pasos:
  3. Estadisticas locales (media/desv. estandar) por imagen en cada solape.
  4. Filtrado de "candidatos de ajuste": se descartan zonas de reflexion especular
     (media local baja -> agua, asfalto liso) y zonas de doble rebote
     (desv. estandar local alta -> zonas urbanas).
  5. Construccion y resolucion del sistema global de minimos cuadrados (ridge)
     para obtener un factor de correccion multiplicativo h_i por imagen.
  6. Aplicacion de h_i a cada imagen y exportacion, mas tabla de metricas
     antes/despues (Dmu, Ddelta, RMSE) igual al Table II del paper.

Requiere: numpy, scipy, rasterio
"""

from __future__ import annotations

import itertools
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import uniform_filter


# --------------------------------------------------------------------------
# Configuracion
# --------------------------------------------------------------------------

@dataclass
class Config:
    window_size: int = 9          # ventana para estadisticas locales (impar)
    low_mean_pct: float = 15.0    # percentil bajo de media -> descartar (especular)
    high_std_pct: float = 85.0    # percentil alto de std -> descartar (doble rebote)
    min_candidates: int = 500     # minimo de pixeles candidatos validos por par
    ridge_lambda: float = 1e-3    # regularizacion del sistema de minimos cuadrados
    nodata_value: float | None = 0.0  # valor a tratar como nodata (ajustar si aplica)


# --------------------------------------------------------------------------
# Paso 3: lectura + estadisticas locales
# --------------------------------------------------------------------------

def local_mean_std(arr: np.ndarray, window: int, valid_mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Media y desviacion estandar local via filtro de ventana deslizante,
    excluyendo pixeles invalidos (nodata) del calculo."""
    arr_filled = np.where(valid_mask, arr, 0.0)
    valid_f = valid_mask.astype(np.float64)

    count = uniform_filter(valid_f, size=window, mode="nearest")
    count = np.clip(count, 1e-6, None)

    mean = uniform_filter(arr_filled, size=window, mode="nearest") / count
    mean_sq = uniform_filter(arr_filled ** 2, size=window, mode="nearest") / count
    var = np.clip(mean_sq - mean ** 2, 0, None)
    std = np.sqrt(var)
    return mean, std


def overlap_arrays_on_grid_i(path_i: str, path_j: str, cfg: Config):
    """Reproyecta la imagen j sobre la grilla de la imagen i, recorta al bbox
    de solape y devuelve ambos arrays alineados pixel a pixel."""
    with rasterio.open(path_i) as src_i, rasterio.open(path_j) as src_j:
        bounds_i = src_i.bounds
        bounds_j = src_j.bounds

        left = max(bounds_i.left, bounds_j.left)
        right = min(bounds_i.right, bounds_j.right)
        bottom = max(bounds_i.bottom, bounds_j.bottom)
        top = min(bounds_i.top, bounds_j.top)

        if left >= right or bottom >= top:
            return None  # sin solape

        window_i = rasterio.windows.from_bounds(left, bottom, right, top, transform=src_i.transform)
        window_i = window_i.round_offsets().round_lengths()
        if window_i.width < 2 or window_i.height < 2:
            return None

        arr_i = src_i.read(1, window=window_i).astype(np.float64)
        transform_overlap = src_i.window_transform(window_i)

        arr_j = np.full(arr_i.shape, np.nan, dtype=np.float64)
        reproject(
            source=rasterio.band(src_j, 1),
            destination=arr_j,
            src_transform=src_j.transform,
            src_crs=src_j.crs,
            dst_transform=transform_overlap,
            dst_crs=src_i.crs,
            resampling=Resampling.bilinear,
        )

        return arr_i, arr_j


# --------------------------------------------------------------------------
# Paso 4: seleccion de candidatos (excluir especular y doble rebote)
# --------------------------------------------------------------------------

def select_candidates(arr_i, arr_j, cfg: Config):
    """Filtro de 'candidatos de ajuste' del paper: descarta reflexion especular
    (media local baja) y doble rebote (desv. estandar local alta) en ambas
    imagenes, quedandose solo con terreno rugoso estable."""
    nodata = cfg.nodata_value
    valid = np.isfinite(arr_i) & np.isfinite(arr_j)
    if nodata is not None:
        valid &= (arr_i != nodata) & (arr_j != nodata)
    valid &= (arr_i > 0) & (arr_j > 0)  # sigma0 lineal debe ser positivo

    if valid.sum() < cfg.min_candidates:
        return None

    mean_i, std_i = local_mean_std(arr_i, cfg.window_size, valid)
    mean_j, std_j = local_mean_std(arr_j, cfg.window_size, valid)

    low_i = np.percentile(mean_i[valid], cfg.low_mean_pct)
    low_j = np.percentile(mean_j[valid], cfg.low_mean_pct)
    high_i = np.percentile(std_i[valid], cfg.high_std_pct)
    high_j = np.percentile(std_j[valid], cfg.high_std_pct)

    stable = (
        valid
        & (mean_i > low_i) & (mean_j > low_j)   # descarta especular
        & (std_i < high_i) & (std_j < high_j)   # descarta doble rebote
    )

    if stable.sum() < cfg.min_candidates:
        return None

    return arr_i[stable], arr_j[stable]


# --------------------------------------------------------------------------
# Paso 5: sistema global de minimos cuadrados (ganancias h_i)
# --------------------------------------------------------------------------

@dataclass
class PairStats:
    idx_i: int
    idx_j: int
    log_ratio: float   # log( mediana(sigma_j) / mediana(sigma_i) ) sobre candidatos
    n_candidates: int
    mean_i: float
    mean_j: float
    std_i: float
    std_j: float


def solve_gains(pairs: list[PairStats], n_images: int, cfg: Config) -> np.ndarray:
    """Resuelve log(h_i) - log(h_j) = log_ratio_ij para todas las parejas a la vez
    (ajuste conjunto, no en cadena), con regularizacion ridge para fijar el
    nivel de referencia (el sistema es indeterminado ante un desplazamiento
    global constante de todos los log(h_i))."""
    rows = []
    rhs = []
    for p in pairs:
        row = np.zeros(n_images)
        row[p.idx_i] = 1.0
        row[p.idx_j] = -1.0
        rows.append(row)
        rhs.append(p.log_ratio)

    A = np.array(rows)
    L = np.array(rhs)

    AtA = A.T @ A + cfg.ridge_lambda * np.eye(n_images)
    AtL = A.T @ L
    log_h = np.linalg.solve(AtA, AtL)
    log_h -= log_h.mean()  # ancla: referencia = nivel medio del conjunto
    return np.exp(log_h)


def build_pairs(image_paths: list[str], cfg: Config) -> list[PairStats]:
    pairs = []
    for (i, path_i), (j, path_j) in itertools.combinations(enumerate(image_paths), 2):
        res = overlap_arrays_on_grid_i(path_i, path_j, cfg)
        if res is None:
            continue
        arr_i, arr_j = res
        cand = select_candidates(arr_i, arr_j, cfg)
        if cand is None:
            continue
        vals_i, vals_j = cand
        ratio = np.median(vals_j) / np.median(vals_i)
        if not np.isfinite(ratio) or ratio <= 0:
            continue
        pairs.append(PairStats(
            idx_i=i, idx_j=j,
            log_ratio=float(np.log(ratio)),
            n_candidates=int(vals_i.size),
            mean_i=float(vals_i.mean()), mean_j=float(vals_j.mean()),
            std_i=float(vals_i.std()), std_j=float(vals_j.std()),
        ))
    return pairs


# --------------------------------------------------------------------------
# Paso 6: aplicar ganancias, exportar, y metricas antes/despues
# --------------------------------------------------------------------------

def apply_gain_and_save(path_in: str, gain: float, path_out: str):
    with rasterio.open(path_in) as src:
        arr = src.read(1).astype(np.float64)
        profile = src.profile.copy()
        corrected = arr * gain
        profile.update(dtype="float32", nodata=0.0)
        with rasterio.open(path_out, "w", **profile) as dst:
            dst.write(corrected.astype(np.float32), 1)


def compute_metrics(pairs: list[PairStats], gains: np.ndarray, label: str):
    """Replica los indicadores del Table II del paper (Dmu, Ddelta, RMSE),
    calculados antes (gains=1) y despues de aplicar las ganancias."""
    d_mu, d_delta, sq_errors = [], [], []
    for p in pairs:
        m_i = p.mean_i * gains[p.idx_i]
        m_j = p.mean_j * gains[p.idx_j]
        s_i = p.std_i * gains[p.idx_i]
        s_j = p.std_j * gains[p.idx_j]
        d_mu.append(abs(m_i - m_j))
        d_delta.append(abs(s_i - s_j))
        sq_errors.append((m_i - m_j) ** 2)
    d_mu = float(np.mean(d_mu)) if d_mu else float("nan")
    d_delta = float(np.mean(d_delta)) if d_delta else float("nan")
    rmse = float(np.sqrt(np.mean(sq_errors))) if sq_errors else float("nan")
    return {"label": label, "D_mu": d_mu, "D_delta": d_delta, "RMSE": rmse}


# --------------------------------------------------------------------------
# Orquestacion
# --------------------------------------------------------------------------

def run(image_paths: list[str], output_dir: str, cfg: Config | None = None):
    cfg = cfg or Config()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"[1/4] Construyendo pares de solape y candidatos para {len(image_paths)} imagenes...")
    pairs = build_pairs(image_paths, cfg)
    if not pairs:
        raise RuntimeError(
            "No se encontraron pares con solape valido y suficientes candidatos. "
            "Revisa CRS/bounds de los GeoTIFF o baja 'min_candidates'."
        )
    print(f"      {len(pairs)} pares utilizables de {len(image_paths) * (len(image_paths) - 1) // 2} posibles.")

    connected = {p.idx_i for p in pairs} | {p.idx_j for p in pairs}
    isolated = [image_paths[i] for i in range(len(image_paths)) if i not in connected]
    if isolated:
        print("      AVISO: estas imagenes no tuvieron NINGUN par de solape valido")
        print("      y quedaran sin corregir de forma confiable:")
        for p in isolated:
            print(f"        - {p}")

    print("[2/4] Resolviendo sistema global de ganancias (minimos cuadrados ridge)...")
    gains = solve_gains(pairs, len(image_paths), cfg)
    for path, g in zip(image_paths, gains):
        print(f"      h[{Path(path).name}] = {g:.4f}")

    print("[3/4] Calculando metricas antes/despues...")
    gains_before = np.ones_like(gains)
    metrics_before = compute_metrics(pairs, gains_before, "antes (original)")
    metrics_after = compute_metrics(pairs, gains, "despues (RPRN)")
    for m in (metrics_before, metrics_after):
        print(f"      {m['label']:22s} D_mu={m['D_mu']:.4f}  D_delta={m['D_delta']:.4f}  RMSE={m['RMSE']:.4f}")

    print("[4/4] Aplicando ganancias y exportando imagenes corregidas...")
    out_paths = []
    for path, g in zip(image_paths, gains):
        out_path = output_dir / f"{Path(path).stem}_RN.tif"
        apply_gain_and_save(path, g, str(out_path))
        out_paths.append(str(out_path))
        print(f"      -> {out_path}")

    report = {
        "gains": {Path(p).name: float(g) for p, g in zip(image_paths, gains)},
        "n_pairs_used": len(pairs),
        "isolated_images": isolated,
        "metrics_before": metrics_before,
        "metrics_after": metrics_after,
        "output_images": out_paths,
    }
    report_path = output_dir / "rprn_report.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\nReporte guardado en: {report_path}")
    return report


if __name__ == "__main__":
    TIFF_DIR = "C:/argenis/RADAR/TIFF"
    IMAGE_PATHS = [str(p) for p in Path(TIFF_DIR).glob("*.tif")]
    OUTPUT_DIR = "C:/argenis/RADAR/CALIB2"
    run(IMAGE_PATHS, OUTPUT_DIR)

[1/4] Construyendo pares de solape y candidatos para 12 imagenes...
      19 pares utilizables de 66 posibles.
[2/4] Resolviendo sistema global de ganancias (minimos cuadrados ridge)...
      h[S1A_IW1_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.9302
      h[S1A_IW1_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.8714
      h[S1A_IW1_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif] = 1.0212
      h[S1A_IW1_SLC__1SDV_20180830T101056_023474_028E3B_0188.tif] = 1.8828
      h[S1A_IW2_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.6675
      h[S1A_IW2_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.7610
      h[S1A_IW2_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif] = 1.1035
      h[S1A_IW2_SLC__1SDV_20180830T101056_023474_028E3B_0188.tif] = 1.2633
      h[S1A_IW3_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.7365
      h[S1A_IW3_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.9085
      h[S1A_IW3_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif